# 06 JEV and LAYA as judges
JEV and LAYA never hear the audio: they only reason over the evidence we send.

- **Section 1, JEV on one detector:** can JEV improve a single detector? FP1 sends one number (D2's score) plus the
  threshold and D2's own verdict, so JEV can only repeat it: 100% coverage, 0 corrected errors (FP1 runs on 2021
  DF). Below, JEV gets richer evidence and is checked against simple rules that see the same evidence. Saved results;
  not rerun.
- **Section 2, LAYA as an arbiter between detectors:** our detectors often disagree. Can LAYA (local, free) pick the
  right one? Every request and answer can be inspected.

## Section 1: JEV on one detector's evidence

In [2]:
import hashlib
import json
from pathlib import Path
import librosa
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from audio_df.data.manifests import load_manifest
from audio_df.helpers.audio import load_audio
from audio_df.helpers.eval_audio import TRIM_RECIPE
from audio_df.helpers.metrics import compute_eer
from audio_df.helpers.research.runs import collect
from audio_df.modules.jev import JEVState, decide

SR, SETS, N_RANDOM, N_DISAGREE = 16000, ["itw", "asvspoof2021_hidden", "asvspoof2021df_hidden"], 10, 10
DETECTORS = {"a1": "A1 LFCC-GMM", "a2": "A2 WavLM", "d2 L6": "D2 WavLM+LFCC one-way", "d3 L6": "D3 WavLM+LFCC two-way"}
runs = {k[0]: v["scores"] for k, v in collect("../runs", ["a1", "a2", "d2", "d3"]).items()
        if k[1] == "trimmed" and k[0] in DETECTORS}

def platt(dev):  # dev score -> P(bona fide)
    return LogisticRegression().fit(dev[["score"]], dev.label == "bonafide")

calib = {m: platt(runs[m]["dev"]) for m in DETECTORS}
thr = {m: compute_eer(runs[m]["dev"].score, runs[m]["dev"].label, higher_is_spoof=False)[1] for m in DETECTORS}

d:\MyCode\projects\GC2-EchoJEV\SystemCode\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.1 Clips
Per test set: 10 random clips plus 10 where D2 and A2 disagree at their dev thresholds, so there are mistakes to fix.
Results are reported per group; the disagreement group is deliberately hard.

In [3]:
rng = np.random.default_rng(1337)
pool = []
for s in SETS:
    merged = None
    for m in DETECTORS:
        part = runs[m][s][["clip_id", "label", "score"]].rename(columns={"score": m})
        merged = part if merged is None else merged.merge(part, on=["clip_id", "label"])
    merged["set"] = s
    disagree = (merged["d2 L6"] <= thr["d2 L6"]) != (merged["a2"] <= thr["a2"])
    pick_d = merged[disagree].sample(N_DISAGREE, random_state=1337).assign(group="disagree")
    pick_r = merged[~merged.clip_id.isin(pick_d.clip_id)].sample(N_RANDOM, random_state=1337).assign(group="random")
    pool += [pick_r, pick_d]
pool = pd.concat(pool, ignore_index=True)
for m in DETECTORS:
    pool[f"p_{m}"] = calib[m].predict_proba(pool[[m]].rename(columns={m: "score"}))[:, 1]
pool.groupby(["set", "group"]).label.value_counts().unstack(fill_value=0)

label                           bonafide  spoof
set                   group                    
asvspoof2021_hidden   disagree         5      5
                      random           0     10
asvspoof2021df_hidden disagree         1      9
                      random           0     10
itw                   disagree        10      0
                      random           9      1

## 1.2 Readable audio facts
What an analyst could read off the waveform: duration, leading silence (same rule as trimming), effective bandwidth
(highest frequency within 50 dB of the spectral peak: ~8 kHz for studio audio, ~4 kHz for telephone or narrowband)
and share of speech energy above 4 kHz (spoofs had less in notebook 03).

In [4]:
paths = {s: load_manifest(f"../data/manifests/{s}.csv", "..").set_index("utt_id").path for s in SETS}

def describe(path):
    x = load_audio(path, SR)
    speech = librosa.effects.split(x, top_db=TRIM_RECIPE["top_db"], frame_length=TRIM_RECIPE["frame_length"],
                                   hop_length=TRIM_RECIPE["hop_length"])
    voiced = np.concatenate([x[s:e] for s, e in speech]) if len(speech) else x
    power = (np.abs(librosa.stft(voiced, n_fft=512)) ** 2).mean(axis=1)  # average spectrum of speech
    freqs = librosa.fft_frequencies(sr=SR, n_fft=512)
    return {"duration_s": round(len(x) / SR, 2),
            "leading_silence_s": round(speech[0, 0] / SR, 2) if len(speech) else 0.0,
            # highest frequency within 50 dB of the spectral peak: ~8 kHz studio, ~4 kHz telephone/narrowband
            "bandwidth_khz": round(float(freqs[np.nonzero(power >= power.max() * 1e-5)[0].max()]) / 1000, 2),
            "energy_above_4khz_pct": round(100 * power[freqs >= 4000].sum() / power.sum(), 2)}

pool = pool.join(pd.DataFrame([describe(paths[r.set][r.clip_id]) for r in pool.itertuples()]))
pool.groupby("label")[["duration_s", "leading_silence_s", "bandwidth_khz", "energy_above_4khz_pct"]].median()

,duration_s,leading_silence_s,bandwidth_khz,energy_above_4khz_pct
label,,,,
bonafide,2.85,0.13,7.69,0.35
spoof,1.74,0.00,7.72,0.43


## 1.3 Rules without JEV, on the same clips
- **D2 alone:** its dev threshold.
- **WavLM average:** mean P(bona fide) of A2, D2, D3 ≥ 0.5.
- **Abstain on disagreement:** D2's verdict, or "uncertain" when D2 and A2 disagree.

In [5]:
def summarise(pred, keep=None):  # pred: "bonafide" / "spoof" / "uncertain" per clip; keep: rows to score
    keep = np.ones(len(pool), bool) if keep is None else keep
    pred, label = np.asarray(pred)[keep], pool.label.to_numpy()[keep]
    decided = pred != "uncertain"
    correct = (pred == label) & decided
    return pd.Series({"coverage": decided.mean(), "accuracy on decided": correct.sum() / max(decided.sum(), 1),
                      "errors": int((decided & ~correct).sum()), "abstained": int((~decided).sum())})

verdict = lambda p: np.where(p >= 0.5, "bonafide", "spoof")
d2 = np.where(pool["d2 L6"] > thr["d2 L6"], "bonafide", "spoof")
a2 = np.where(pool["a2"] > thr["a2"], "bonafide", "spoof")
policies = {"D2 alone": d2,
            "WavLM average": verdict(pool[["p_a2", "p_d2 L6", "p_d3 L6"]].mean(axis=1)),
            "abstain on disagreement": np.where(d2 == a2, d2, "uncertain")}
pd.DataFrame({k: summarise(v) for k, v in policies.items()}).T.round(3)

,coverage,accuracy on decided,errors,abstained
D2 alone,1.000,0.617,23.0,0.0
WavLM average,1.000,0.650,21.0,0.0
abstain on disagreement,0.467,0.821,5.0,32.0


## 1.4 JEV with richer evidence
Each request carries every detector's calibrated P(bona fide), what we measured about each detector's weak spots,
and the audio facts above. It never sees the dataset name, clip ID or label. Raw D2 tokens or attention weights are
not sent: they are learned 128-number projections with no meaning a reader (or an LLM) can map to acoustics.

`RUN_JEV = False` keeps this free; set it to True to call the API (one paid call per clip not already cached in
`data/cache/jev/notebook06/`).

In [6]:
RUN_JEV = False
CACHE = Path("../data/cache/jev/notebook06")
GUIDANCE = (
    "You see evidence about one speech clip from four spoofing detectors and simple audio measurements; you do not "
    "hear the audio. Detector reliability measured on unseen web audio and newer spoofing attacks: A2 and D2 are the "
    "most reliable (~10-20% error); D3 is similar on studio audio but less reliable on web audio; A1 is unreliable "
    "on compressed or web audio (~30-58% error). bandwidth_khz near 8 means studio-quality audio; below ~4.5 means "
    "telephone or narrowband audio, where the LFCC-based A1 is least trustworthy. Leading silence is not evidence "
    "either way. Answer 'uncertain' when the reliable detectors disagree or are near 0.5.")
QUESTIONS = {
    "decision": {"type": "choice", "instructions": "Which verdict does the evidence support?",
                 "criteria": {"bonafide": "Reliable evidence favours authentic speech.",
                              "spoof": "Reliable evidence favours spoofed speech.",
                              "uncertain": "Reliable detectors conflict or are inconclusive."}},
    "evidence_quality": {"type": "score", "instructions": "How consistent is the reliable evidence?",
                         "criteria": ["Conflicting", "Mixed", "Consistent"]},
}

FACTS = ["duration_s", "leading_silence_s", "bandwidth_khz", "energy_above_4khz_pct"]

def context(r):  # evidence for one clip; no dataset name, clip id or label
    return {**{f"p_bonafide_{name.split()[0]}": round(float(r[f"p_{m}"]), 3) for m, name in DETECTORS.items()},
            **{k: float(r[k]) for k in FACTS}}

def ask(ctx):
    payload = {"context": ctx, "prompt": GUIDANCE, "questions": QUESTIONS}
    path = CACHE / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest() + ".json")
    if path.exists():
        return json.loads(path.read_text())
    if not RUN_JEV:
        return None
    answers = decide(JEVState(context=ctx, prompt=GUIDANCE, questions=QUESTIONS)).answers
    answers = answers if isinstance(answers, dict) else json.loads(json.dumps(answers, default=lambda o: o.__dict__))
    CACHE.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(answers))
    return answers

replies = [ask(context(r)) for r in pool.to_dict("records")]
print(f"{sum(x is not None for x in replies)}/{len(replies)} JEV answers available")

0/60 JEV answers available


## 1.5 Comparison
Same clips for every policy, per group: random clips and the hard disagreement clips.

In [7]:
if all(x is not None for x in replies):
    policies["JEV (rich evidence)"] = np.array([x["decision"]["choice"] for x in replies])
pd.concat({g: pd.DataFrame({k: summarise(v, (pool.group == g).to_numpy()) for k, v in policies.items()}).T
           for g in ["random", "disagree"]}).round(3)

coverage  accuracy on decided  errors  \
random   D2 alone                    1.000                0.800     6.0   
         WavLM average               1.000                0.733     8.0   
         abstain on disagreement     0.933                0.821     5.0   
disagree D2 alone                    1.000                0.433    17.0   
         WavLM average               1.000                0.567    13.0   
         abstain on disagreement     0.000                0.000     0.0   

                                  abstained  
random   D2 alone                       0.0  
         WavLM average                  0.0  
         abstain on disagreement        2.0  
disagree D2 alone                       0.0  
         WavLM average                  0.0  
         abstain on disagreement       30.0

### Section 1 result
- **One detector's evidence:** JEV can only repeat the detector (FP1: 100% coverage, nothing corrected).
- **JEV with richer evidence:** answers were not cached in this saved run (0/60), so there is no JEV row in 1.5.
- **The disagreements carry the signal.** On these clips, abstaining whenever D2 and A2 disagree lifts accuracy on
  decided clips from 0.62 (D2 alone) to 0.82, at 47% coverage. The disagreement itself says where the errors are.
  Section 2 tries to resolve those disagreements instead of only abstaining.

## Section 2: LAYA as an arbiter between detectors
The web app's compare view puts several detectors on one clip. Here LAYA gets every detector's verdict and decides
which to trust. The test is `uv run arbitration` ([arbitration.py](../src/audio_df/pipelines/arbitration.py)).

The test case is fixed in that file:
- **Detectors:** A1 LFCC-GMM, A2 WavLM and A4 LFCC-MLP, the latest runs trained on trimmed audio. Each uses the dev
  EER threshold of its own artifact.
- **Probabilities and reliability:** P(bona fide) comes from Platt scaling on dev. Reliability is each detector's
  EER on 2021 LA eval, a different set from the one tested here.
- **Clips:** `11249.wav` (bona fide; A1 and A4 say bona fide, A2 says spoof) plus 200 In-the-Wild clips where the
  three do not all agree.

This section reads the newest `runs/E-*-arbitration` run. Only 2.1 and 2.2 call LAYA live (local and free).
`--arbiter jev` runs the same test on JEV (paid).

In [2]:
from audio_df.modules import laya
from audio_df.pipelines.arbitration import ORDERS, VARIANTS, build_cases, choice, load_detectors, request

ARBITER = laya.decide  # jev.decide accepts the same request (needs TYPESAFE_API_KEY, paid per call)
arb_run = sorted(Path("../runs").glob("E-*-arbitration"))[-1]
cases = pd.read_csv(arb_run / "cases.csv")
arb_metrics = json.loads((arb_run / "metrics.json").read_text())
det = load_detectors(Path(".."))
print(f"{arb_run.name}: {arb_metrics['arbiter']} on {arb_metrics['n_cases']} {arb_metrics['testset']} clips")
pd.DataFrame(arb_metrics["detectors"]).T

E-20261028-151028-393865-arbitration: laya on 201 itw clips


,name,run,threshold,error_rate_pct
a1,A1 LFCC-GMM,runs/E-20261023-172139-766981-a1,0.738177,34.1
a2,A2 WavLM,runs/E-20261023-200932-976565-a2,-0.002726,14.3
a4,A4 LFCC-MLP,runs/E-20261023-194755-458184-a4,-0.252304,40.6


### 2.1 The request
What LAYA sees depends on the variant; `order` sets the order the detectors are listed in. The variants are:
- **verdicts:** each detector's verdict only.
- **probabilities:** each verdict plus its P(bona fide).
- **reliability:** the above plus each detector's measured error rate, and an instruction to trust lower error.
- **reliability order:** detectors anonymised and listed from most to least reliable.
- **combined:** one reliability-weighted P(bona fide) plus the vote count, so LAYA only has to read one number.

The request below is the reliability variant for `11249.wav`, with LAYA's live answer.

In [3]:
clip = cases[cases.clip_id == "ITW_11249"].iloc[0]
req = request(clip, det, "reliability")
print(json.dumps({"prompt": req.prompt, "context": dict(req.context)}, indent=1))
print(json.dumps(ARBITER(req)["answers"]["decision"], indent=1))

{
 "prompt": "Several spoofing detectors judged the same clip and disagree. Decide which verdict to trust. Trust detectors with a lower error_rate_pct more.",
 "context": {
  "A1 LFCC-GMM": {
   "verdict": "bonafide",
   "p_bonafide": 0.688,
   "error_rate_pct": 34.1
  },
  "A2 WavLM": {
   "verdict": "spoof",
   "p_bonafide": 0.001,
   "error_rate_pct": 14.3
  },
  "A4 LFCC-MLP": {
   "verdict": "bonafide",
   "p_bonafide": 0.154,
   "error_rate_pct": 40.6
  }
 }
}



Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 1795.97it/s]

/Users/zuttome/LocalDocs/Learning/College/NUS/MTech-AIS/PS/GC2/SystemCode/.venv/lib/python3.11/site-packages/laya/agent.py:2157: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


{
 "type": "choice",
 "choice": "bonafide",
 "probabilities": {
  "bonafide": 0.4929,
  "spoof": 0.3042,
  "uncertain": 0.2029
 },
 "confidence": 0.0585,
 "answer_confidence": 0.4929,
 "action": {
  "act_probability": 1.0
 }
}


### 2.2 Inspect one clip: `11249.wav` (bona fide)
The same clip under every variant and both orders. A judge that reads the evidence should not change its answer
when only the listing order changes.

In [4]:
print({det[m]["name"]: clip[f"v_{m}"] for m in det}, "| truth:", clip.label)
pd.DataFrame([{"variant": v, "order": o, "choice": (a := ARBITER(request(clip, det, v, order))["answers"]["decision"])["choice"],
               **a["probabilities"]} for v in VARIANTS for o, order in ORDERS.items()]).round(3)

{'A1 LFCC-GMM': 'bonafide', 'A2 WavLM': 'spoof', 'A4 LFCC-MLP': 'bonafide'} | truth: bonafide


,variant,order,choice,bonafide,spoof,uncertain
0,verdicts,A1 first,spoof,0.245,0.690,0.065
1,verdicts,A1 last,spoof,0.057,0.895,0.048
2,probabilities,A1 first,bonafide,0.586,0.341,0.073
3,probabilities,A1 last,spoof,0.093,0.861,0.046
4,reliability,A1 first,bonafide,0.493,0.304,0.203
5,reliability,A1 last,spoof,0.086,0.835,0.078
6,reliability order,A1 first,spoof,0.133,0.785,0.082
7,reliability order,A1 last,spoof,0.133,0.785,0.082
8,combined,A1 first,uncertain,0.412,0.100,0.489
9,combined,A1 last,uncertain,0.412,0.100,0.489


### 2.3 All clips: does LAYA read the evidence?
Per variant: how often the answer flips when only the listing order changes, how often it says uncertain, and how
often it agrees with each detector.

In [5]:
probe = []
for v in VARIANTS:
    first, last = cases[f"{v} | A1 first"], cases[f"{v} | A1 last"]
    probe.append({"variant": v, "order flips": (first != last).mean(), "uncertain": (first == "uncertain").mean(),
                  "answers bonafide": (first == "bonafide").mean(),
                  **{f"agrees with {m.upper()} (A1 first / last)":
                     f"{(first == cases[f'v_{m}']).mean():.2f} / {(last == cases[f'v_{m}']).mean():.2f}" for m in det}})
pd.DataFrame(probe).set_index("variant").round(2)

,order flips,uncertain,answers bonafide,agrees with A1 (A1 first / last),agrees with A2 (A1 first / last),agrees with A4 (A1 first / last)
variant,,,,,,
verdicts,0.00,0.00,0.00,0.73 / 0.73,0.64 / 0.64,0.14 / 0.14
probabilities,0.47,0.00,0.26,0.99 / 0.51,0.44 / 0.89,0.33 / 0.36
reliability,0.40,0.00,0.26,0.99 / 0.61,0.41 / 0.82,0.33 / 0.07
reliability order,0.00,0.00,0.01,0.72 / 0.72,0.65 / 0.65,0.15 / 0.15
combined,0.00,0.85,0.15,0.02 / 0.02,0.12 / 0.12,0.11 / 0.11


### 2.4 Accuracy against simple rules (same clips)

In [6]:
pd.DataFrame(arb_metrics["results"]).T.round(3)

,coverage,accuracy_on_decided,errors
A1 LFCC-GMM alone,1.000,0.333,134.0
A2 WavLM alone,1.000,0.786,43.0
A4 LFCC-MLP alone,1.000,0.438,113.0
majority vote,1.000,0.557,89.0
reliability-weighted P >= 0.5,1.000,0.617,77.0
LAYA verdicts | A1 first,1.000,0.428,115.0
LAYA verdicts | A1 last,1.000,0.428,115.0
LAYA probabilities | A1 first,1.000,0.328,135.0
LAYA probabilities | A1 last,1.000,0.672,66.0
LAYA reliability | A1 first,1.000,0.318,137.0


### 2.5 Confusion matrices
Rows are the true label; columns are LAYA's answer. Each detector alone is shown for reference. In-the-Wild is
mostly bona fide, so read the bona fide row first.

In [7]:
ANSWERS = ["bonafide", "spoof", "uncertain"]
confusion = {**{f"{det[m]['name']} alone": cases[f"v_{m}"] for m in det},
             **{f"LAYA {c}": cases[c] for c in cases.columns if " | " in c}}
pd.concat({name: pd.crosstab(cases.label, pred).reindex(columns=ANSWERS, fill_value=0)
           for name, pred in confusion.items()}).rename_axis(["rule", "truth"])

bonafide  spoof  uncertain
rule                              truth                               
A1 LFCC-GMM alone                 bonafide        18     97          0
                                  spoof           37     49          0
A2 WavLM alone                    bonafide        72     43          0
                                  spoof            0     86          0
A4 LFCC-MLP alone                 bonafide        87     28          0
                                  spoof           85      1          0
LAYA verdicts | A1 first          bonafide         0    115          0
                                  spoof            0     86          0
LAYA verdicts | A1 last           bonafide         0    115          0
                                  spoof            0     86          0
LAYA probabilities | A1 first     bonafide        16     99          0
                                  spoof           36     50          0
LAYA probabilities | A1 last      bonafide        49     66          0
                                  spoof            0     86          0
LAYA reliability | A1 first       bonafide        15    100          0
                                  spoof           37     49          0
LAYA reliability | A1 last        bonafide        35     80          0
                                  spoof            0     86          0
LAYA reliability order | A1 first bonafide         2    113          0
                                  spoof            0     86          0
LAYA reliability order | A1 last  bonafide         2    113          0
                                  spoof            0     86          0
LAYA combined | A1 first          bonafide        26      0         89
                                  spoof            5      0         81
LAYA combined | A1 last           bonafide        26      0         89
                                  spoof            5      0         81

### Section 2 result
LAYA does not arbitrate on the evidence it is given. It leans towards spoof, and otherwise copies the detector
listed first.
- **Order changes the answer.** Listing the same detectors in a different order flips 47% of answers with
  probabilities and 40% with reliability notes. With A1 listed first LAYA agrees with A1 on 99% of clips, and its
  confusion matrix is almost A1's own (bona fide 16 right / 99 wrong vs A1's 18 / 97). With A2 listed first it agrees
  with A2 on 82–89%.
- **Its errors are bona fide called spoof.** With verdicts only it answers spoof on every clip. Listed by
  reliability, it answers spoof on 199 of 201. Even its best setup (probabilities, A2 first) catches all 86 spoofs
  but calls 66 of 115 bona fide clips spoof.
- **It does not abstain on per-detector evidence.** No "uncertain" answers, although every clip is a disagreement.
- **The best LAYA setup is still below A2 alone:** 0.67 accuracy against A2's 0.79. Most variants are below a
  majority vote (0.56).
- **Combined evidence turns LAYA into a filter, not a judge.** Given one weighted probability, it abstains on 85% of
  clips and never answers spoof. Of the 31 clips it calls bona fide, 26 are bona fide.

On `11249.wav` (bona fide), LAYA says bona fide only when A1 is listed first with probabilities or reliability notes.
Otherwise it says spoof, siding with A2. That is the "prefers SSL" behaviour seen in the app: the listing order, not
the evidence, decides.

**How to use LAYA or JEV here.** The arbitration itself should be a rule we can check, such as reliability weights
fitted on dev or a learned combiner. LAYA or JEV can then explain that decision, or flag low-confidence cases for
review. A caution for this detector set: on In-the-Wild, A2 alone beats every combination, because A1 and A4 are
wrong more often than right when they disagree with it. Arbitration only helps when the other detectors add
independent, reliable evidence.